In [6]:
import numpy as np
from numpy.random import rand, randint
import gurobipy as gp
from gurobipy import GRB
from TakeTwoSetup import *

In [7]:
# --- Finding D and H --- #
# D: max number of days
# H: max number of scenes in a day

D = (2*n*(max(durations)+np.matrix.max(S)))/(W+np.matrix.max(S))
if np.ceil(D)%2 == 0:
  D = int(np.floor(D))
else:
  D=int(np.ceil(D))

increasingDurations = np.sort(durations)
increasingChangeover = np.array(np.sort(np.matrix.flatten(S)))[0]

vals = []
hs = []
for h in range(2, n+1):
  total = np.sum(increasingChangeover[:h]) + np.sum(increasingDurations[:h])
  if total <= W:
    # to make sure combination is within admissible set
    vals.append(total)
    hs.append(h)
H = hs[np.argmax(vals)]

In [8]:
# --- ILP_pos Model Setup --- #

ILP_pos = gp.Model("ILP_pos", env=env)

# decision variables
x_pos = ILP_pos.addVars(n, H, D, vtype=GRB.BINARY, name="x_pos")
y_pos = ILP_pos.addVars(n, n, H-1, D, vtype=GRB.BINARY, name="y_pos")

# aux variable (tau)
tau = ILP_pos.addVars(m, 2, lb=1, ub=D, vtype=GRB.INTEGER, name="tau")

# objective function
objFunc_pos = ILP_pos.setObjective(
    gp.quicksum(
        holdingCost[i] * (tau[i, 1] - tau[i, 0] + 1)
        for i in range(m)
    ), GRB.MINIMIZE
)

# --- constraints (they aint gonna be pretty) o7 --- #
c5 = ILP_pos.addConstrs((
    gp.quicksum (
        x_pos[j, r, d]
        for d in range(D)
        for r in range(H)
    ) == 1
    for j in range(n)),
    name="c5"
) # Constraint 5

c6 = ILP_pos.addConstrs((
    gp.quicksum(
        x_pos[j, r, d]
        for j in range(n)
    ) <= 1
    for d in range(D)
    for r in range(H)),
    name = "c6"
) # Constraint 6

c7 = ILP_pos.addConstrs((
    gp.quicksum(
        x_pos[j, r+1, d]
        for j in range(n)
    ) <=
    gp.quicksum(
        x_pos[j, r, d]
        for j in range(n)
    )
    for d in range(D)
    for r in range(H-1)),
    name="c7"
) # Constraint 7

c8 = ILP_pos.addConstrs(
    (gp.quicksum(
        x_pos[j, 1, d+1]
        for j in range(n)
    ) <=
    gp.quicksum(
        x_pos[j, 1, d]
        for j in range(n)
    ) for d in range(D-1)),
    name="c8"
) # Constraint 8

c9 = ILP_pos.addConstrs((
    tau[i, 0] <= gp.quicksum(
        d * x_pos[j, r, d]
        for r in range(H)
        for d in range(D)
    )for i in range(m)
     for j in range(n)
     if T[i,j]),
    name="c9"
) # Constraint 9

c10 = ILP_pos.addConstrs((
    gp.quicksum(
        d * x_pos[j, r, d]
        for r in range(H)
        for d in range(D)
    ) <= tau[i, 1]
     for i in range(m)
     for j in range(n)
     if T[i,j]),
    name="c10"
) # Constraint 10

c11 = ILP_pos.addConstrs((
    x_pos[j1, r, d] + x_pos[j2, r+1, d] <= y_pos[j1, j2, r, d] + 1
    for d in range(D)
    for j1 in range(n)
    for j2 in range(n)
    for r in range(H-1)
    if j1 != j2),
    name = "c11"
) #Constraint 11

c12 = ILP_pos.addConstrs((
    gp.quicksum(
        y_pos[j1, j2,r,d] + y_pos[j2,j1,r,d]
        for r in range(H-1)
    )<= 1
    for j1 in range(n)
    for j2 in range(n)
    for d in range (D)
    if j1 != j2),
    name = "c12"
) #Constraint 12

c13 = ILP_pos.addConstrs((
    gp.quicksum(
        durations[j] * x_pos[j,r,d]
        for r in range(H)
        for j in range(n)
    ) + gp.quicksum(
        S[j1,j2]*y_pos[j1,j2,r,d]
        for r in range(H-1)
        for j1 in range(n)
        for j2 in range(n)
        if j1 != j2
    )<= W
    for d in range (D)),
    name = "c13"
) #Constraint 13

# --- Constraints (14-16) --- #
# These constraints are embedded in the init of the variables

In [9]:
ILP_pos.optimize()
ILP_pos.getJSONSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: AMD Ryzen 7 4700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2773076 - for non-commercial use only - registered to km___@uga.edu
Optimize a model with 2765 rows, 2529 columns and 16038 nonzeros (Min)
Model fingerprint: 0x5877a7c2
Model has 18 linear objective coefficients and an objective constant of 6970
Variable types: 0 continuous, 2529 integer (2511 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [6e+02, 9e+02]
  Bounds range     [1e+00, 9e+00]
  RHS range        [1e+00, 1e+02]

Presolve removed 2664 rows and 2439 columns
Presolve time: 0.01s
Presolved: 101 rows, 90 columns, 816 nonzeros
Variable types: 0 continuous, 90 integer (72 binary)

Root relaxation: infeasible, 107 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    

'{ "SolutionInfo": { "Status": 3, "Runtime": 1.1599993705749512e-01, "Work": 1.8335632480606406e-02, "ObjBound": 1e+100, "ObjBoundC": 1e+100, "MIPGap": 1e+100, "IterCount": 107, "BarIterCount": 0, "NLBarIterCount": 0, "PDHGIterCount": 0, "NodeCount": 1, "SolCount": 0}}'

In [10]:
all_vars = ILP_pos.getVars()
values = ILP_pos.getAttr("X", all_vars)
names = ILP_pos.getAttr("VarName", all_vars)

for name, val in zip(names, values):
    if val == 1:
        print(f"{name} = {val}")

GurobiError: Unable to retrieve attribute 'X'